In [1]:
import os
import re
import time
import subprocess
import coverage
from tqdm import tqdm

In [2]:
# === CONFIG ===
TEST_DIR = "RQ3_SBERT_HNSW_Prompt2_testscripts"
NUM_TESTS = 599
SOURCE_DIR = "."                  
COVERAGE_SUFFIX = ".coverage_" 

In [3]:
total_tests = 0
passed = 0
failed = 0
coverage_results = []

In [4]:
def get_coverage_metrics(cov_data_file):
    """Accurately calculate line and branch coverage from a coverage data file."""
    cov = coverage.Coverage(data_file=cov_data_file)
    cov.load()

    total_statements = 0
    total_missing = 0
    total_branches = 0
    total_missing_branches = 0

    for filename in cov.get_data().measured_files():
        try:
            # Line-level analysis
            analysis = cov.analysis2(filename)
            total_statements += len(analysis[1])
            total_missing += len(analysis[2])

            # Branch-level analysis (semi-private API)
            summary = cov._analyze(filename)
            total_branches += summary.numbers.n_branches
            total_missing_branches += summary.numbers.n_missing_branches

        except Exception as e:
            print(f"[WARN] Could not analyze {filename}: {e}")

    line_cov = 100.0 * (total_statements - total_missing) / total_statements if total_statements else 0.0
    branch_cov = 100.0 * (total_branches - total_missing_branches) / total_branches if total_branches else 0.0

    return {
        "line_coverage": line_cov,
        "branch_coverage": branch_cov
    }

print("🧪 Running test files with coverage tracking...")

🧪 Running test files with coverage tracking...


In [5]:
# === Initialize Tracking ===
total_tests = 0
passed = 0
failed = 0
coverage_results = []

print("🧹 Cleaning test files before coverage run...")
start_time = time.time()

🧹 Cleaning test files before coverage run...


In [6]:
start_time = time.time()

for i in tqdm(range(NUM_TESTS), desc="Running tests with coverage", unit="file"):
    test_file = os.path.join(TEST_DIR, f"test_code_{i}.py")
    if not os.path.exists(test_file):
        print(f"❌ {test_file} not found, skipping.")
        continue

    cov_data_file = f".coverage_{i}"

    # Run unittest with coverage tracking and 2s timeout
    cmd = [
        "coverage", "run",
        f"--data-file={cov_data_file}",
        "--branch",
        f"--source={SOURCE_DIR}",
        "-m", "unittest", test_file
    ]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=2)
        output = result.stdout + result.stderr

        print(f"\n▶️ Output from {test_file}:\n{output}")

        # Extract test count
        m = re.search(r"Ran (\d+) tests?", output)
        tests_run = int(m.group(1)) if m else 0
        total_tests += tests_run

        if "OK" in output and "FAILED" not in output:
            passed += tests_run
        else:
            fail = int(re.search(r"failures=(\d+)", output).group(1)) if re.search(r"failures=(\d+)", output) else 0
            err = int(re.search(r"errors=(\d+)", output).group(1)) if re.search(r"errors=(\d+)", output) else 0
            failed += (fail + err)
            passed += (tests_run - fail - err)

        # Collect coverage data
        metrics = get_coverage_metrics(cov_data_file)
        coverage_results.append({
            "test_file": test_file,
            "line_coverage": metrics["line_coverage"],
            "branch_coverage": metrics["branch_coverage"]
        })

    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout: {test_file} took more than 2 seconds and was skipped.")
        failed += 1
        coverage_results.append({
            "test_file": test_file,
            "line_coverage": 0.0,
            "branch_coverage": 0.0
        })
    except Exception as e:
        print(f"⚠️ Error running {test_file}: {e}")
        failed += 1
        coverage_results.append({
            "test_file": test_file,
            "line_coverage": 0.0,
            "branch_coverage": 0.0
        })
    finally:
        if os.path.exists(cov_data_file):
            os.remove(cov_data_file)

end_time = time.time()
runtime = round(end_time - start_time, 2)

print(f"\n✅ Finished coverage run in {runtime} seconds")
print(f"Total Tests Run: {total_tests} | Passed: {passed} | Failed: {failed}")

Running tests with coverage:   0%|                                                                | 1/599 [00:00<04:19,  2.31file/s]


▶️ Output from RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_0.py:
........
----------------------------------------------------------------------
Ran 8 tests in 0.001s

OK



Running tests with coverage:   0%|▏                                                               | 2/599 [00:00<04:42,  2.11file/s]


▶️ Output from RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_1.py:
FFFFFFF
FAIL: test_example_1 (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_1.TestSolution.test_example_1)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_1.py", line 20, in test_example_1
    self.assertEqual(result, [0, 1])
AssertionError: Lists differ: [1, 0] != [0, 1]

First differing element 0:
1
0

- [1, 0]
+ [0, 1]

FAIL: test_example_2 (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_1.TestSolution.test_example_2)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_1.py", line 26, in test_example_2
    self.assertEqual(result, [1, 2])
AssertionError: Lists differ: [2, 1] != [1, 2]

First differing 

Running tests with coverage:   1%|▎                                                               | 3/599 [00:01<04:38,  2.14file/s]

...


▶️ Output from RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_596.py:
.....FF.
FAIL: test_large_piles (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_596.TestSolution.test_large_piles)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_596.py", line 51, in test_large_piles
    self.assertEqual(self.solution.mergeStones([10, 20, 30, 40, 50, 60, 70, 80, 90, 100], 2), 945)
AssertionError: 1730 != 945

FAIL: test_minimum_piles_merge (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_596.TestSolution.test_minimum_piles_merge)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_596.py", line 45, in test_minimum_piles_merge
    self.assertEqual(self.solution.mergeStones([4, 6, 4, 7], 2

Running tests with coverage: 100%|█████████████████████████████████████████████████████████████▉| 598/599 [05:59<00:00,  1.38file/s]


▶️ Output from RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_597.py:
.EF.F
ERROR: test_invalid_ips_empty_logs (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_597.TestSolution.test_invalid_ips_empty_logs)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_597.py", line 42, in test_invalid_ips_empty_logs
    result = self.solution.find_invalid_ips(logs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_597.py", line 17, in find_invalid_ips
    ].sort_values(['invalid_count', 'ip'], ascending=[0, 0])
      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\test_gen\Lib\site-packages\pandas\core\frame.py", line 7179, in sort_values
    keys = [self._get_label_or_level_values(x,

Running tests with coverage: 100%|██████████████████████████████████████████████████████████████| 599/599 [05:59<00:00,  1.66file/s]


▶️ Output from RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_598.py:
....FF..
FAIL: test_kth_smallest_path_larger_grid (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_598.TestSolution.test_kth_smallest_path_larger_grid)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_598.py", line 50, in test_kth_smallest_path_larger_grid
    self.assertEqual(self.solution.kthSmallestPath([3, 3], 10), "VHHHVV")
AssertionError: 'HVVVHH' != 'VHHHVV'
- HVVVHH
+ VHHHVV


FAIL: test_kth_smallest_path_larger_k (RQ3_SBERT_HNSW_Prompt2_testscripts.test_code_598.TestSolution.test_kth_smallest_path_larger_k)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "E:\3. SUMMER 2025\summer 25\Unit Testing\new_samples\RQ3_SBERT_HNSW_Prompt2_testscripts\test_code_598.py", line 53, in test_kth_smalles

In [7]:
# === Report Summary ===
avg_line_cov = sum(r["line_coverage"] for r in coverage_results) / len(coverage_results) if coverage_results else 0
avg_branch_cov = sum(r["branch_coverage"] for r in coverage_results) / len(coverage_results) if coverage_results else 0

print("\n📊 Coverage Summary:")
print(f"Total Test Cases Run: {total_tests}")
print(f"Tests Passed: {passed}")
print(f"Tests Failed/Errored: {failed}")
print(f"Average Line Coverage: {avg_line_cov:.2f}%")
print(f"Average Branch Coverage: {avg_branch_cov:.2f}%")
print(f"Total Runtime: {runtime}s")


📊 Coverage Summary:
Total Test Cases Run: 4567
Tests Passed: 3971
Tests Failed/Errored: 608
Average Line Coverage: 98.00%
Average Branch Coverage: 80.86%
Total Runtime: 359.83s
